# 7.1 Knowledge, logic and propositional logic: practice notebook

Build an agent that knows facts and works out new ones. Meet the wumpus world, learn what models and entailment mean, write sentences in propositional logic, and check entailment with a truth table.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### The truth table of the connectives

In [ ]:
# The truth table of the five connectives. True and False are written T and F.
def show(v):
    return "T" if v else "F"

print("P Q | not P | P and Q | P or Q | P => Q | P <=> Q")
for p in [True, False]:
    for q in [True, False]:
        implies = (not p) or q          # false only when P is true and Q is false
        iff = (p == q)                  # true when both have the same value
        print(show(p), show(q), "|  ", show(not p), "  |   ", show(p and q), "   |  ", show(p or q), "   |  ",
              show(implies), "   |   ", show(iff))

### Entailment by listing models

In [ ]:
# Model checking: KB = {P or Q, P => R, Q => R}. Does the KB entail R? Does it entail P?
from itertools import product

def kb_true(p, q, r):
    return (p or q) and ((not p) or r) and ((not q) or r)

models = []
for p, q, r in product([True, False], repeat=3):     # all 8 models
    if kb_true(p, q, r):
        models.append((p, q, r))
print("models of the KB (P, Q, R):", models)

entails_r = all(r for p, q, r in models)             # R true in every model of the KB?
entails_p = all(p for p, q, r in models)
print("KB entails R:", entails_r)
print("KB entails P:", entails_p)

## Your turn

### Task 1: Evaluate a sentence in a model

Sentences are written as Python values. A symbol is a string such as `"P"`. A complex sentence is a tuple: `("not", s)`, `("and", a, b)`, `("or", a, b)`, `("implies", a, b)` or `("iff", a, b)`. A model is a dictionary from symbols to True or False. Write `value(s, model)`.

Hint: A string is looked up in the model. For a tuple, look at the first item and compute the parts with value itself.

In [ ]:
def value(s, model):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("value('P', {'P': True})", True), ("value(('implies', 'P', 'Q'), {'P': True, 'Q': False})", False), ("value(('implies', 'P', 'Q'), {'P': False, 'Q': False})", True), ("value(('iff', 'B11', ('or', 'P12', 'P21')), {'B11': False, 'P12': False, 'P21': False})", True), ("value(('not', ('and', 'P', 'Q')), {'P': True, 'Q': True})", False)])

### Task 2: Count the models of the wumpus KB

The starter gives the wumpus knowledge base R1 to R5, `value` and `all_models(symbols)`. Write `count_models(kb, symbols)`: the number of models in which every sentence of kb is true.

Hint: Loop over all_models(symbols). A model counts when value(s, m) is True for every s in kb.

In [ ]:
SYMBOLS = ["B11", "B21", "P11", "P12", "P21", "P22", "P31"]
KB = [("not", "P11"),
      ("iff", "B11", ("or", "P12", "P21")),
      ("iff", "B21", ("or", "P11", ("or", "P22", "P31"))),
      ("not", "B11"),
      "B21"]

def value(s, model):
    if type(s) == str:
        return model[s]
    op = s[0]
    if op == "not":
        return not value(s[1], model)
    a = value(s[1], model)
    b = value(s[2], model)
    if op == "and":
        return a and b
    if op == "or":
        return a or b
    if op == "implies":
        return (not a) or b
    return a == b

def all_models(symbols):
    # every model as a dictionary, made by doubling the list once per symbol
    models = [{}]
    for name in symbols:
        bigger = []
        for m in models:
            t = dict(m)
            t[name] = True
            f = dict(m)
            f[name] = False
            bigger.append(t)
            bigger.append(f)
        models = bigger
    return models

def count_models(kb, symbols):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('count_models(KB, SYMBOLS)', 3), ("count_models([('or', 'P', 'Q')], ['P', 'Q', 'R'])", 6), ("count_models([('or', 'P', 'Q'), ('implies', 'P', 'R'), ('implies', 'Q', 'R')], ['P', 'Q', 'R'])", 3), ("count_models([('and', 'P', ('not', 'P'))], ['P'])", 0)])

### Task 3: TT-ENTAILS for the wumpus world

Write `tt_entails(kb, alpha, symbols)`: True if alpha is true in every model where all sentences of kb are true. Use `all_models` and `value` from the starter.

Hint: Look for a counterexample: a model where the KB is true and alpha is false. If there is none, return True.

In [ ]:
SYMBOLS = ["B11", "B21", "P11", "P12", "P21", "P22", "P31"]
KB = [("not", "P11"),
      ("iff", "B11", ("or", "P12", "P21")),
      ("iff", "B21", ("or", "P11", ("or", "P22", "P31"))),
      ("not", "B11"),
      "B21"]

def value(s, model):
    if type(s) == str:
        return model[s]
    op = s[0]
    if op == "not":
        return not value(s[1], model)
    a = value(s[1], model)
    b = value(s[2], model)
    if op == "and":
        return a and b
    if op == "or":
        return a or b
    if op == "implies":
        return (not a) or b
    return a == b

def all_models(symbols):
    # every model as a dictionary, made by doubling the list once per symbol
    models = [{}]
    for name in symbols:
        bigger = []
        for m in models:
            t = dict(m)
            t[name] = True
            f = dict(m)
            f[name] = False
            bigger.append(t)
            bigger.append(f)
        models = bigger
    return models

def tt_entails(kb, alpha, symbols):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("tt_entails(KB, ('not', 'P12'), SYMBOLS)", True), ("tt_entails(KB, 'P22', SYMBOLS)", False), ("tt_entails(KB, ('not', 'P22'), SYMBOLS)", False), ("tt_entails(KB, ('or', 'P22', 'P31'), SYMBOLS)", True), ("tt_entails([('implies', 'P', 'Q'), 'Q'], 'P', ['P', 'Q'])", False)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Evaluate a sentence in a model

The fourth test is R2 of the wumpus KB in a model with no breeze and no pits: both sides are false, so ⇔ is true.

In [ ]:
def value(s, model):
    if type(s) == str:
        return model[s]
    op = s[0]
    if op == "not":
        return not value(s[1], model)
    a = value(s[1], model)
    b = value(s[2], model)
    if op == "and":
        return a and b
    if op == "or":
        return a or b
    if op == "implies":
        return (not a) or b
    if op == "iff":
        return a == b

In [ ]:
check([("value('P', {'P': True})", True), ("value(('implies', 'P', 'Q'), {'P': True, 'Q': False})", False), ("value(('implies', 'P', 'Q'), {'P': False, 'Q': False})", True), ("value(('iff', 'B11', ('or', 'P12', 'P21')), {'B11': False, 'P12': False, 'P21': False})", True), ("value(('not', ('and', 'P', 'Q')), {'P': True, 'Q': True})", False)])

### Solution to task 2: Count the models of the wumpus KB

The wumpus KB has 3 models out of 128, as in the textbook.

In [ ]:
SYMBOLS = ["B11", "B21", "P11", "P12", "P21", "P22", "P31"]
KB = [("not", "P11"),
      ("iff", "B11", ("or", "P12", "P21")),
      ("iff", "B21", ("or", "P11", ("or", "P22", "P31"))),
      ("not", "B11"),
      "B21"]

def value(s, model):
    if type(s) == str:
        return model[s]
    op = s[0]
    if op == "not":
        return not value(s[1], model)
    a = value(s[1], model)
    b = value(s[2], model)
    if op == "and":
        return a and b
    if op == "or":
        return a or b
    if op == "implies":
        return (not a) or b
    return a == b

def all_models(symbols):
    # every model as a dictionary, made by doubling the list once per symbol
    models = [{}]
    for name in symbols:
        bigger = []
        for m in models:
            t = dict(m)
            t[name] = True
            f = dict(m)
            f[name] = False
            bigger.append(t)
            bigger.append(f)
        models = bigger
    return models

def count_models(kb, symbols):
    count = 0
    for m in all_models(symbols):
        ok = True
        for s in kb:
            if not value(s, m):
                ok = False
        if ok:
            count = count + 1
    return count

In [ ]:
check([('count_models(KB, SYMBOLS)', 3), ("count_models([('or', 'P', 'Q')], ['P', 'Q', 'R'])", 6), ("count_models([('or', 'P', 'Q'), ('implies', 'P', 'R'), ('implies', 'Q', 'R')], ['P', 'Q', 'R'])", 3), ("count_models([('and', 'P', ('not', 'P'))], ['P'])", 0)])

### Solution to task 3: TT-ENTAILS for the wumpus world

The KB entails ¬P<sub>1,2</sub> and P<sub>2,2</sub> ∨ P<sub>3,1</sub>, but neither P<sub>2,2</sub> nor ¬P<sub>2,2</sub>. The last test is affirming the consequent.

In [ ]:
SYMBOLS = ["B11", "B21", "P11", "P12", "P21", "P22", "P31"]
KB = [("not", "P11"),
      ("iff", "B11", ("or", "P12", "P21")),
      ("iff", "B21", ("or", "P11", ("or", "P22", "P31"))),
      ("not", "B11"),
      "B21"]

def value(s, model):
    if type(s) == str:
        return model[s]
    op = s[0]
    if op == "not":
        return not value(s[1], model)
    a = value(s[1], model)
    b = value(s[2], model)
    if op == "and":
        return a and b
    if op == "or":
        return a or b
    if op == "implies":
        return (not a) or b
    return a == b

def all_models(symbols):
    # every model as a dictionary, made by doubling the list once per symbol
    models = [{}]
    for name in symbols:
        bigger = []
        for m in models:
            t = dict(m)
            t[name] = True
            f = dict(m)
            f[name] = False
            bigger.append(t)
            bigger.append(f)
        models = bigger
    return models

def tt_entails(kb, alpha, symbols):
    for m in all_models(symbols):
        kb_ok = True
        for s in kb:
            if not value(s, m):
                kb_ok = False
        if kb_ok and not value(alpha, m):
            return False
    return True

In [ ]:
check([("tt_entails(KB, ('not', 'P12'), SYMBOLS)", True), ("tt_entails(KB, 'P22', SYMBOLS)", False), ("tt_entails(KB, ('not', 'P22'), SYMBOLS)", False), ("tt_entails(KB, ('or', 'P22', 'P31'), SYMBOLS)", True), ("tt_entails([('implies', 'P', 'Q'), 'Q'], 'P', ['P', 'Q'])", False)])